# YOLOv8 — License Plate Detection (Colab)

Single-class plate detector (`plate`) trained on the Kaggle *License Plate Detection Dataset (10125 images)*, run on a Colab **GPU** runtime, driven from VS Code via the Colab extension.

**Trigger:** Select Kernel → a Colab **GPU** runtime → Run All. TPU will not work (Ultralytics = CUDA).

**Resumable:** checkpoints land on Drive every epoch. After a disconnect, run the **Resume** cell (6b) instead of re-training.

Deliverable: `best.pt` + ONNX in Drive `weights/`; metrics row appended to `src/ml/experiments.csv`.

## 0. Config — edit here, then Run All

In [ ]:
# ── Config ───────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/UIT_2025'                    # checkpoints/weights live here
DATASET_ID  = '1A2p0niaUiJcMi7gd-oULtN0UGREMRVHS'                  # Google Drive file ID of vehicle_plate.zip
DATASET_ZIP = '/content/vehicle_plate.zip'                        # downloaded here by ID (no Drive-path dependency)
DATASET_DIR = '/content/dataset'                                  # normalized unzip target (fast local reads)
RUN_NAME    = 'yolov8n-plate-v1'

MODEL   = 'yolov8n.pt'   # edge target: start n/s, scale up only with evidence
IMGSZ   = 640
EPOCHS  = 10
BATCH   = 16
CLASSES = ['plate']      # single-class license-plate detector

CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints'   # ultralytics writes last.pt/best.pt here every epoch
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

## 1. Setup — pinned installs

In [2]:
!pip install -q ultralytics==8.3.0  # pinned — bump deliberately, never float
import ultralytics
ultralytics.checks()

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 49.1/112.6 GB disk)


## 2. Runtime check — must be GPU

In [3]:
import torch
assert torch.cuda.is_available(), (
    'No CUDA GPU. In VS Code: Select Kernel → a Colab GPU runtime (not TPU/CPU), then Run All.'
)
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## 3. Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
for d in (CKPT_DIR, WEIGHTS_DIR):
    os.makedirs(d, exist_ok=True)
print('Drive mounted. checkpoints →', CKPT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. checkpoints → /content/drive/MyDrive/UIT_2025/checkpoints


## 4. Stage dataset — Drive → local, auto-normalize to `images/<split>` + `labels/<split>`

Kaggle zip layout is not guaranteed (Roboflow `train/valid/test`, flat `images/`+`labels/`, or labels alongside images). This cell auto-detects image↔label pairs, forces every label to class `0` (single-class), and builds a 90/10 train/val split (respecting any existing split dirs).

In [ ]:
import os, glob, zipfile, shutil, random
from pathlib import Path

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
RAW_DIR = '/content/dataset_raw'

# 1. download zip by Drive file ID (skip if already present), then fresh extract
if not os.path.exists(DATASET_ZIP):
    import gdown
    gdown.download(id=DATASET_ID, output=DATASET_ZIP, quiet=False)
print('zip size:', round(os.path.getsize(DATASET_ZIP) / 1e6, 1), 'MB')
if os.path.exists(RAW_DIR):     shutil.rmtree(RAW_DIR)
if os.path.exists(DATASET_DIR): shutil.rmtree(DATASET_DIR)
os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(RAW_DIR)
print('Extracted top-level:', sorted(os.listdir(RAW_DIR))[:20])

# 2. pair each image with its YOLO .txt label (alongside, or images/ -> labels/ sibling)
def find_label(img):
    p = Path(img)
    cand = p.with_suffix('.txt')
    if cand.exists():
        return str(cand)
    parts = list(p.parts)
    for i, part in enumerate(parts):
        if part.lower() in ('images', 'image', 'imgs'):
            alt = Path(*parts[:i], 'labels', *parts[i + 1:]).with_suffix('.txt')
            if alt.exists():
                return str(alt)
    return None

imgs   = [p for p in glob.glob(f'{RAW_DIR}/**/*', recursive=True) if p.lower().endswith(IMG_EXT)]
paired = [(im, find_label(im)) for im in imgs]
paired = [(im, lb) for im, lb in paired if lb]
print(f'images found: {len(imgs)}, with labels: {len(paired)}')
assert paired, 'No YOLO .txt labels found — zip may be Pascal VOC XML; convert before training.'

# 3. assign split: honor existing train/val(id)/test dirs, else random 90/10
def split_of(path):
    low = path.replace(os.sep, '/').lower()
    if '/train/' in low:                                   return 'train'
    if '/val/' in low or '/valid/' in low or '/validation/' in low or '/test/' in low: return 'val'
    return None

random.seed(7)
assigned = [(im, lb, split_of(im) or ('val' if random.random() < 0.10 else 'train'))
            for im, lb in paired]

# 4. copy into normalized layout; force class id 0 (single-class), unique stems to avoid collisions
for split in ('train', 'val'):
    for sub in ('images', 'labels'):
        os.makedirs(f'{DATASET_DIR}/{sub}/{split}', exist_ok=True)

def write_label(src, dst):
    out = []
    for ln in Path(src).read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        parts = ln.split()
        parts[0] = '0'
        out.append(' '.join(parts))
    Path(dst).write_text('\n'.join(out) + ('\n' if out else ''))

counts = {'train': 0, 'val': 0}
for i, (im, lb, s) in enumerate(assigned):
    stem = f'{i:06d}'
    shutil.copy(im, f'{DATASET_DIR}/images/{s}/{stem}{Path(im).suffix.lower()}')
    write_label(lb, f'{DATASET_DIR}/labels/{s}/{stem}.txt')
    counts[s] += 1

print('normalized:', counts)
assert counts['train'] > 0 and counts['val'] > 0, 'Empty split — check extracted layout above.'

zip size: 528.9 MB
Extracted top-level: ['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'test', 'train', 'valid']
images found: 10125, with labels: 10125
normalized: {'train': 7057, 'val': 3068}


## 5. Dataset YAML — inlined (local repo files are not on the runtime)

In [ ]:
import yaml

data_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
DATA_YAML_PATH = '/content/plate-detect.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(open(DATA_YAML_PATH).read())

path: /content/dataset
train: images/train
val: images/val
names:
  0: plate



## 6. Train — checkpoints to Drive every epoch

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML_PATH,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=CKPT_DIR,   # → Drive; last.pt updated every epoch (resume-safe)
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    degrees=10.0,       # small rotation — plates shot at gate angles
    seed=7,
)

New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/plate-detect.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=/content/drive/MyDrive/UIT_2025/checkpoints, name=yolov8n-plate-v1, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=7, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_ma

train: Scanning /content/dataset/labels/train... 7057 images, 5 backgrounds, 0 corrupt: 100%|██████████| 7057/7057 [00:03<00:00, 1905.97it/s]


train: New cache created: /content/dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /content/dataset/labels/val... 3068 images, 4 backgrounds, 0 corrupt: 100%|██████████| 3068/3068 [00:02<00:00, 1187.20it/s]


val: New cache created: /content/dataset/labels/val.cache
Plotting labels to /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1
Starting training for 10 epochs...
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      10/10      2.35G       1.04     0.4851      1.086          2        640: 100%|██████████| 442/442 [02:08<00:00,  3.43it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 96/96 [00:24<00:00,  3.93it/s]

                   all       3068       3280      0.979       0.93      0.965      0.676



10 epochs completed in 0.438 hours.
Optimizer stripped from /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/last.pt, 5.6MB
Optimizer stripped from /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.pt, 5.6MB

Validating /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 96/96 [00:27<00:00,  3.46it/s]


                   all       3068       3280      0.979       0.93      0.964      0.676
Speed: 0.2ms preprocess, 1.9ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1


## 6b. Resume — run ONLY after a disconnect (skip on a clean run)

In [ ]:
# Uncomment + run if training was interrupted. Resumes from Drive last.pt.
# from ultralytics import YOLO
# model = YOLO(f'{CKPT_DIR}/{RUN_NAME}/weights/last.pt')
# model.train(resume=True)

## 7. Validate — per-class metrics

In [ ]:
from ultralytics import YOLO

best = f'{CKPT_DIR}/{RUN_NAME}/weights/best.pt'
metrics = YOLO(best).val(data=DATA_YAML_PATH, imgsz=IMGSZ, split='val')
print('mAP50    :', round(metrics.box.map50, 4))
print('mAP50-95 :', round(metrics.box.map, 4))
print('precision:', round(metrics.box.mp, 4))
print('recall   :', round(metrics.box.mr, 4))

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


val: Scanning /content/dataset/labels/val.cache... 3068 images, 4 backgrounds, 0 corrupt: 100%|██████████| 3068/3068 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 192/192 [00:29<00:00,  6.51it/s]


                   all       3068       3280      0.979       0.93      0.964      0.676
Speed: 0.2ms preprocess, 2.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to runs/detect/val2
mAP50    : 0.9644
mAP50-95 : 0.6763
precision: 0.9787
recall   : 0.9302


## 8. Export ONNX + handoff

In [ ]:
!pip install -q onnx onnxslim onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 48.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.8 MB/s eta 0:00:00


In [ ]:
import shutil, datetime, os
from ultralytics import YOLO

best = f'{CKPT_DIR}/{RUN_NAME}/weights/best.pt'
onnx_path = YOLO(best).export(format='onnx', opset=17, imgsz=IMGSZ, dynamic=False)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
shutil.copy(best, f'{WEIGHTS_DIR}/{RUN_NAME}.pt')
shutil.copy(onnx_path, f'{WEIGHTS_DIR}/{RUN_NAME}.onnx')

# experiments.csv row — runtime cannot write the local repo, so print + paste into src/ml/experiments.csv
row = ','.join(str(x) for x in [
    datetime.date.today().isoformat(), MODEL, os.path.basename(DATASET_ZIP),
    f'imgsz={IMGSZ};epochs={EPOCHS};batch={BATCH}',
    round(metrics.box.map50, 4), round(metrics.box.map, 4),
    round(metrics.box.mp, 4), round(metrics.box.mr, 4),
    f'{WEIGHTS_DIR}/{RUN_NAME}.pt',
])
print('append to src/ml/experiments.csv:')
print(row)

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon 2.00GHz)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.3 MB)

ONNX: starting export with onnx 1.22.0 opset 17...


W0722 17:53:12.017000 1775 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter

ONNX: slimming with onnxslim 0.1.34...
ONNX: export success ✅ 28.0s, saved as '/content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.onnx' (10.4 MB)

Export complete (28.5s)
Results saved to /content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights
Predict:         yolo predict task=detect model=/content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=/content/drive/MyDrive/UIT_2025/checkpoints/yolov8n-plate-v1/weights/best.onnx imgsz=640 data=/content/plate-detect.yaml  
Visualize:       https://netron.app
append to src/ml/experiments.csv:
2026-07-22,yolov8n.pt,vehicle_plate.zip,imgsz=640;epochs=10;batch=16,0.9644,0.6763,0.9787,0.9302,/content/drive/MyDrive/UIT_2025/weights/yolov8n-plate-v1.pt
